In [1]:
# ============================================================
# CHATBOT SOC - VERSIÓN ULTRA-RÁPIDA (OPTIMIZADA PARA GPU)
# ============================================================

# ------------------------------------------------------------
# 1. INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN DE OLLAMA
# ------------------------------------------------------------
print("📦 Instalando dependencias y configurando entorno...")
!pip install -q chromadb sentence-transformers pypdf flask

print("⚙️ Instalando herramientas del sistema (zstd)...")
!apt-get update -qq && apt-get install -y zstd -qq

print("🦙 Configurando Ollama con aceleración por hardware (GPU)...")
!curl -fsSL https://ollama.com/install.sh | sh

# Forzamos a Ollama a utilizar los núcleos CUDA de la GPU T4 de Colab
import os
import subprocess
import time

env_gpu = os.environ.copy()
env_gpu["OLLAMA_NUM_PARALLEL"] = "1"
env_gpu["CUDA_VISIBLE_DEVICES"] = "0"

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env_gpu)
time.sleep(5)  # Esperar a que el servidor de Ollama despierte

# Usamos Llama 3.2 (3B): Excelente rendimiento en español y rápido en GPU T4
MODELO_LLM = "llama3.2"
print(f"📥 Descargando modelo {MODELO_LLM} (esto tomará un par de minutos la primera vez)...")
!ollama pull {MODELO_LLM}

# ------------------------------------------------------------
# 2. DESCARGA DE DOCUMENTOS
# ------------------------------------------------------------
print("📂 Clonando repositorio de datos...")
if not os.path.exists("yahnachee.github.io"):
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/yahna-chee/yahnachee.github.io.git
    %cd yahnachee.github.io
    !git sparse-checkout set IA/soc-chatbot/docs
else:
    %cd yahnachee.github.io

# ------------------------------------------------------------
# 3. LÓGICA RAG (EMBEDDINGS Y BASE DE DATOS)
# ------------------------------------------------------------
print("🤖 Inicializando componentes RAG...")
import requests
import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from pathlib import Path
from flask import Flask, request
from google.colab.output import eval_js
import threading
import warnings
warnings.filterwarnings('ignore')

DOCS_PATH = "IA/soc-chatbot/docs"
OLLAMA_URL = "http://localhost:11434/api/chat"

embedder = SentenceTransformer('all-MiniLM-L6-v2')
cliente = chromadb.Client()

# Eliminar colección previa si existe para evitar conflictos
try:
    cliente.delete_collection("soc_docs")
except:
    pass
coleccion = cliente.create_collection("soc_docs")

print("📄 Procesando e indexando documentos PDF...")
pdfs = list(Path(DOCS_PATH).glob("*.pdf"))

if not pdfs:
    print("⚠️ No se encontraron PDFs. Creando conocimiento de respaldo...")
    coleccion.add(
        embeddings=[embedder.encode("Las fases de respuesta a incidentes en un SOC son: Preparación, Identificación, Contención, Erradicación, Recuperación y Lecciones Aprendidas.").tolist()],
        documents=["Las fases de respuesta a incidentes en un SOC son: Preparación, Identificación, Contención, Erradicación, Recuperación y Lecciones Aprendidas."],
        metadatas=[{"fuente": "manual_soc.pdf", "indice": 0}],
        ids=["default_0"]
    )
else:
    for pdf in pdfs:
        print(f"   📄 Leyendo: {pdf.name}")
        texto = ""
        try:
            reader = PdfReader(pdf)
            for pagina in reader.pages:
                texto += pagina.extract_text() + "\n"
        except Exception as e:
            print(f"   ❌ Error leyendo {pdf.name}: {e}")
            continue

        palabras = texto.split()
        chunks = [" ".join(palabras[i:i+300]) for i in range(0, len(palabras), 250)]

        for i, chunk in enumerate(chunks):
            if chunk.strip():
                emb = embedder.encode(chunk).tolist()
                coleccion.add(
                    embeddings=[emb],
                    documents=[chunk],
                    metadatas=[{"fuente": pdf.name, "indice": i}],
                    ids=[f"{pdf.stem}_{i}"]
                )
    print(f"✅ Documentos indexados correctamente.")

# ------------------------------------------------------------
# 4. FUNCIÓN DE CONSULTA OPTIMIZADA EN VELOCIDAD
# ------------------------------------------------------------
def consultar(pregunta):
    emb_pregunta = embedder.encode(pregunta).tolist()

    # Reducido a los 2 resultados más relevantes para agilizar el procesamiento de contexto
    resultados = coleccion.query(query_embeddings=[emb_pregunta], n_results=2)

    if not resultados['documents'] or not resultados['documents'][0]:
        return "No encontré información relevante en los documentos."

    contexto = "\n\n".join(resultados['documents'][0])
    fuentes = list(set([meta['fuente'] for meta in resultados['metadatas'][0]]))

    prompt = f"""Eres un analista experto en Ciberseguridad y operaciones de SOC.
Responde la pregunta del usuario utilizando ÚNICAMENTE el contexto provisto de los manuales.
Sé claro, profesional y estructura tu respuesta con viñetas si es necesario.

Contexto de los manuales:
{contexto}

Pregunta: {pregunta}

Respuesta técnica (en español):"""

    try:
        respuesta = requests.post(
            OLLAMA_URL,
            json={
                "model": MODELO_LLM,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
                "options": {
                    "temperature": 0.1,
                    "num_predict": 180,  # Limita la respuesta para cortar drásticamente la latencia
                    "num_ctx": 2048      # Ventana de contexto compacta para agilizar la GPU
                }
            }
        )
        if respuesta.status_code == 200:
            return {
                "respuesta": respuesta.json()["message"]["content"].strip(),
                "fuentes": fuentes
            }
        return f"Error en Ollama: {respuesta.status_code}"
    except Exception as e:
        return f"Error de conexión: {e}"

# ------------------------------------------------------------
# 5. INTERFAZ WEB (FLASK) Y LANZAMIENTO
# ------------------------------------------------------------
app = Flask(__name__)

@app.route('/')
def home():
    return f'''
    <!DOCTYPE html>
    <html>
    <head>
        <title>🛡️ SOC Chatbot</title>
        <meta charset="utf-8">
    </head>
    <body style="font-family:sans-serif; background:#0a0e17; color:#e0e6ed; padding:40px; text-align:center;">
        <div style="background:#1a2a4a; padding:30px; border-radius:12px; max-width:500px; margin:auto; border:1px solid #1e3a5f;">
            <h2>🛡️ SOC Chatbot v2</h2>
            <p style="color:#4ade80;">● Modelo acelerado por GPU: {MODELO_LLM}</p>
            <form action="/ask" method="get">
                <input type="text" name="q" placeholder="¿Qué deseas consultar?" style="width:80%; padding:10px; border-radius:6px; border:none;" required><br><br>
                <button type="submit" style="padding:10px 20px; background:#60a5fa; border:none; border-radius:6px; color:white; font-weight:bold; cursor:pointer;">Preguntar</button>
            </form>
        </div>
    </body>
    </html>
    '''

@app.route('/ask')
def ask():
    pregunta = request.args.get('q', '')
    resultado = consultar(pregunta)

    if isinstance(resultado, dict):
        res_html = resultado["respuesta"].replace('\n', '<br>')
        fuentes = ", ".join(resultado["fuentes"])
        return f'''
        <body style="font-family:sans-serif; background:#0a0e17; color:#e0e6ed; padding:40px;">
            <div style="background:#1a2a4a; padding:30px; border-radius:12px; max-width:600px; margin:auto; border:1px solid #1e3a5f;">
                <h3 style="color:#60a5fa;">Resultados:</h3>
                <p><strong>Pregunta:</strong> {pregunta}</p>
                <div style="background:#0f1829; padding:15px; border-radius:6px; border:1px solid #1e3a5f; text-align:left; line-height:1.6;">{res_html}</div>
                <p><small>📂 Fuentes consultadas: {fuentes}</small></p>
                <a href="/" style="color:#60a5fa; text-decoration:none;">↩ Volver a consultar</a>
            </div>
        </body>
        '''
    return f"<h3>Error</h3><p>{resultado}</p><a href='/'>Volver</a>"

# Iniciar servidor Flask en segundo plano
threading.Thread(target=app.run, kwargs={'host':'0.0.0.0', 'port':5000, 'debug':False}, daemon=True).start()
time.sleep(2)

# Generar enlace tunnel de Colab
url = eval_js("google.colab.kernel.proxyPort(5000)")
print("\n" + "="*50)
print(f"🌐 ENLACE PÚBLICO DEL CHATBOT: {url}")
print("="*50)

📦 Instalando dependencias y configurando entorno...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.5 MB/s eta 0:00

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📄 Procesando e indexando documentos PDF...
   📄 Leyendo: 03_data_theft_playbook.pdf
   📄 Leyendo: 05_denial_of_service_playbook.pdf
   📄 Leyendo: 02_phishing_playbook.pdf
   📄 Leyendo: 08_root_access_playbook.pdf
   📄 Leyendo: 01_malware_outbreak_playbook.pdf
   📄 Leyendo: 07_elevation_of_privilege_playbook.pdf
   📄 Leyendo: 06_unauthorized_access_playbook.pdf
   📄 Leyendo: 04_virus_outbreak_playbook.pdf
   📄 Leyendo: 09_improper_usage_playbook.pdf
✅ Documentos indexados correctamente.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



🌐 ENLACE PÚBLICO DEL CHATBOT: https://5000-gpu-t4-s-kkb-usw4a2-1xoq6shmlt0ou-a.us-west4-2.prod.colab.dev
